## Sel 1: Impor Pustaka & Setup Awal

In [ ]:
import pandas as pd
import geopandas as gpd
import libpysal
from esda.moran import Moran, Moran_Local, Moran_BV, Moran_Local_BV
import mapclassify as mc
import numpy as np
import json
import os
import re
import warnings
import gdown # Tambahan untuk mengunduh dari Google Drive

warnings.filterwarnings('ignore')

# Seed tetap agar reproducible
np.random.seed(42)

# Buat folder output dan data jika belum ada
os.makedirs('output', exist_ok=True)
os.makedirs('data', exist_ok=True)

# Fungsi normalisasi kode wilayah sesuai aturan BPS
def normalisasi_kode(kode):
    if pd.isna(kode):
        return np.nan
    kode_str = str(kode)
    # Jika terbaca sebagai float (misal 3301.0), buang .0 di akhir
    if kode_str.endswith('.0'):
        kode_str = kode_str[:-2]
    # Buang semua karakter non-digit (misal titik pada 33.01)
    kode_bersih = re.sub(r'\D', '', kode_str)
    # Kembalikan dengan panjang minimal 4 karakter
    return kode_bersih.zfill(4)

## Sel 2: Inspeksi, Normalisasi, & Gabung Data Panel

In [ ]:
# ID File Excel
id_padi = '1IgLz2692IW8gZNGWFIxOZRL-V7tWCSFt'
id_penduduk = '1q5ID0tMMaO3B24wR9LqNPMCXWnUMt9vA'
id_kerja = '1b20VLNsDTCpUvcj-iDfg_GSd95PNfMCQ'

def proses_tabel_gdrive(file_id, output_filename, nama_value):
    output_path = f'data/{output_filename}.xlsx'

    if not os.path.exists(output_path):
        print(f"Mengunduh {output_filename} dari Google Drive...")
        url = f'https://drive.google.com/uc?id={file_id}'
        gdown.download(url, output_path, quiet=False)

    df = pd.read_excel(output_path, header=1)

    kolom = list(df.columns)
    kolom[0] = 'kode_bps'
    kolom[1] = 'nama_wilayah'
    df.columns = [str(k) for k in kolom]

    kolom_tahun = [col for col in df.columns if col.isdigit()]

    df_long = pd.melt(df, id_vars=['kode_bps', 'nama_wilayah'],
                      value_vars=kolom_tahun, var_name='tahun', value_name=nama_value)

    df_long['kode_kabkota'] = df_long['kode_bps'].apply(normalisasi_kode)
    df_long['tahun'] = df_long['tahun'].astype(int)

    # Buang baris provinsi
    df_long = df_long[~df_long['kode_kabkota'].str.endswith('00')]
    return df_long[['kode_kabkota', 'tahun', nama_value]]

# Eksekusi (kalori tidak diikutkan, satuan padi diubah jadi kg)
df_padi = proses_tabel_gdrive(id_padi, 'produksi_padi', 'produksi_padi_kg')
df_penduduk = proses_tabel_gdrive(id_penduduk, 'jumlah_penduduk', 'penduduk_jiwa')
df_kerja = proses_tabel_gdrive(id_kerja, 'penduduk_kerja', 'penduduk_kerja_pertanian')

# Merge data panel (tanpa kalori)
df_panel = df_padi.merge(df_penduduk, on=['kode_kabkota', 'tahun'], how='outer')
df_panel = df_panel.merge(df_kerja, on=['kode_kabkota', 'tahun'], how='outer')

df_panel = df_panel[(df_panel['tahun'] >= 2020) & (df_panel['tahun'] <= 2025)]
df_panel = df_panel[df_panel['kode_kabkota'].str.startswith('33')]

print("Dimensi data panel:", df_panel.shape)

## Sel 3: Baca & Prapemrosesan Data Geospasial (SHP)

In [ ]:
import zipfile
import glob
import os

# Masukkan ID File Google Drive dari file ZIP SHP Anda
id_shp_zip = 'ISI_ID_FILE_ZIP_SHP_DISINI' # Pastikan ID ini sudah terisi

zip_path = 'data/kab_kota_shp.zip'
extract_path = 'data/shp_extracted'

# 1. Unduh file ZIP dari Google Drive jika belum ada
if not os.path.exists(zip_path):
    print("Mengunduh ZIP SHP dari Google Drive...")
    url_shp = f'https://drive.google.com/uc?id={id_shp_zip}'
    gdown.download(url_shp, zip_path, quiet=False)

# 2. Ekstrak file ZIP
print("Mengekstrak file ZIP...")
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)

# 3. Cari file .shp secara otomatis di dalam folder hasil ekstraksi
shp_files = glob.glob(f'{extract_path}/**/*.shp', recursive=True)

assert len(shp_files) > 0, "File .shp tidak ditemukan di dalam ZIP! Pastikan ZIP berisi file shapefile."
if len(shp_files) > 1:
    print(f"PERINGATAN: Ditemukan lebih dari satu file .shp. Skrip akan menggunakan yang pertama: {shp_files[0]}")

path_shp = shp_files[0]
print(f"Membaca file SHP: {path_shp}")
gdf = gpd.read_file(path_shp)

print("CRS Awal:", gdf.crs)
kolom_kode_shp = 'KODE_KK'
kolom_nama_shp = 'KAB_KOTA'

# Simpan contoh sebelum normalisasi
sampel_shp_asli = gdf[kolom_kode_shp].head().tolist()

# Normalisasi kode SHP
gdf['kode_kabkota'] = gdf[kolom_kode_shp].apply(normalisasi_kode)

# Cek 5 contoh normalisasi SHP
print("\nContoh Normalisasi Kode SHP (Asli -> Hasil):")
for asli, hasil in zip(sampel_shp_asli, gdf['kode_kabkota'].head()):
    print(f"  {asli} -> {hasil}")

# Filter Jateng (Awalan 33)
gdf_jateng = gdf[gdf['kode_kabkota'].str.startswith('33')]

# Dissolve untuk mengatasi baris ganda (multi-poligon yang terpisah barisnya)
gdf_jateng = gdf_jateng.dissolve(by='kode_kabkota', as_index=False)
gdf_jateng['nama_kabkota'] = gdf_jateng[kolom_nama_shp] # Ambil kembali namanya

# Pengecekan Himpunan Kode SHP vs Panel
kode_unik_shp = set(gdf_jateng['kode_kabkota'].unique())
print(f"\nJumlah kab/kota di SHP Jateng: {len(kode_unik_shp)}")

# --- PERBAIKAN: Definisikan kode_unik_panel dari df_panel di sini ---
kode_unik_panel = set(df_panel['kode_kabkota'].unique())
# --------------------------------------------------------------------

kode_kurang = kode_unik_panel - kode_unik_shp
kode_lebih = kode_unik_shp - kode_unik_panel

if len(kode_kurang) > 0:
    print(f"KODE ADA DI PANEL TAPI HILANG DI SHP: {kode_kurang}")
if len(kode_lebih) > 0:
    print(f"KODE ADA DI SHP TAPI TIDAK ADA DI PANEL: {kode_lebih}")

assert len(kode_unik_shp) == 35, f"SHP tidak memiliki tepat 35 wilayah! Ditemukan: {len(kode_unik_shp)}"
assert kode_unik_shp == kode_unik_panel, "Himpunan kode kab/kota di SHP dan Data Panel tidak sama persis!"

print("\nDaftar Kab/Kota Terpilih (Cek Manual):")
for idx, row in gdf_jateng.iterrows():
    print(f"- {row['kode_kabkota']}: {row['nama_kabkota']}")

# Reproyeksi ke EPSG:4326 (WGS84) jika belum
if gdf_jateng.crs.to_string() != 'EPSG:4326':
    gdf_jateng = gdf_jateng.to_crs(epsg=4326)

## Sel 4: Variabel Turunan & Bobot Spasial

In [ ]:
# Karena data Excel sudah dalam 'kg', kita langsung bagi dengan jumlah penduduk
df_panel['padi_per_kapita'] = df_panel['produksi_padi_kg'] / df_panel['penduduk_jiwa']

# Cek total produksi (Sanity check)
total_produksi_2025 = df_panel[df_panel['tahun'] == 2025]['produksi_padi_kg'].sum()
print(f"Total produksi padi Jateng 2025 (Sanity Check): {total_produksi_2025 / 1e9:.2f} Juta Ton")

# Buat Bobot Spasial Queen Contiguity
gdf_jateng = gdf_jateng.sort_values('kode_kabkota').reset_index(drop=True)
w = libpysal.weights.Queen.from_dataframe(gdf_jateng, use_index=False)
w.transform = 'r'

## Sel 5: Moran's I Global & LISA per Tahun

In [ ]:
df_panel['padi_per_kapita'] = pd.to_numeric(df_panel['padi_per_kapita'], errors='coerce')
df_panel['penduduk_kerja_pertanian'] = pd.to_numeric(df_panel['penduduk_kerja_pertanian'], errors='coerce')

# Isi NaN dengan rata-rata agar Moran's I tidak error (jika ada)
df_panel['padi_per_kapita'] = df_panel['padi_per_kapita'].fillna(df_panel['padi_per_kapita'].mean())

hasil_moran_global = []
data_per_tahun = []

def get_kuadran_lisa(q, p_sim, alpha=0.05):
    if p_sim >= alpha: return "Tidak signifikan"
    return {1: "HH", 2: "LH", 3: "LL", 4: "HL"}.get(q, "Tidak signifikan")

for tahun in range(2020, 2026):
    df_t = df_panel[df_panel['tahun'] == tahun].sort_values('kode_kabkota').reset_index(drop=True)
    x_padi = df_t['padi_per_kapita'].values

    # Hanya Univariat Padi
    mi_padi = Moran(x_padi, w, permutations=999)

    hasil_moran_global.append({
        'tahun': tahun,
        'moran_padi_I': mi_padi.I, 'moran_padi_p': mi_padi.p_sim, 'moran_padi_z': mi_padi.z_sim
    })

    lisa_padi = Moran_Local(x_padi, w, permutations=999, seed=42)

    for idx, row in df_t.iterrows():
        lag_padi = libpysal.weights.lag_spatial(w, x_padi)[idx]
        z_padi = (x_padi[idx] - x_padi.mean()) / x_padi.std()

        data_per_tahun.append({
            'kode_kabkota': row['kode_kabkota'],
            'tahun': tahun,
            'produksi_padi_kg': row['produksi_padi_kg'],
            'padi_per_kapita': row['padi_per_kapita'],
            'penduduk_kerja_pertanian': row['penduduk_kerja_pertanian'],
            'lisa_uni': get_kuadran_lisa(lisa_padi.q[idx], lisa_padi.p_sim[idx]),
            'lisa_uni_p': lisa_padi.p_sim[idx],
            'z_padi': z_padi,
            'lag_padi': lag_padi
        })

df_output = pd.DataFrame(data_per_tahun)
print("Sel 5 Selesai: Moran's I dan LISA (Univariat) berhasil dihitung!")

In [ ]:
from IPython.display import display

# 1. Ringkasan Moran's I Global (2020-2025)
df_moran = pd.DataFrame(hasil_moran_global)
print("=== 1. Ringkasan Global Moran's I (2020-2025) ===")
print("Tips Membaca: Jika nilai p (p_sim) < 0.05, berarti ada pengelompokan (cluster) spasial yang signifikan.")
display(df_moran.round(4))

# 2. Cek Cluster LISA Univariat (Padi per Kapita) untuk tahun terakhir (2025)
tahun_cek = 2025
df_lisa_padi = df_output[(df_output['tahun'] == tahun_cek) &
                         (df_output['lisa_uni'] != 'Tidak signifikan')]

print(f"\n=== 2. Wilayah dengan Cluster Ketersediaan Padi SIGNIFIKAN (Tahun {tahun_cek}) ===")
print("- HH (High-High) : Lumbung padi dikelilingi lumbung padi")
print("- LL (Low-Low)   : Wilayah minim padi dikelilingi wilayah minim padi (biasanya urban/industri)")
print("- LH (Low-High)  : Wilayah minim padi tapi dikelilingi daerah lumbung")
print("- HL (High-Low)  : Wilayah lumbung yang dikelilingi daerah minim padi")

df_lisa_padi_nama = df_lisa_padi.merge(gdf_jateng[['kode_kabkota', 'nama_kabkota']], on='kode_kabkota', how='left')
display(df_lisa_padi_nama[['nama_kabkota', 'padi_per_kapita', 'lisa_uni', 'lisa_uni_p']].sort_values('lisa_uni'))

## Sel 6: Klasifikasi Choropleth & Skala Domain (Pooled)

In [ ]:
# 7. Kumpulkan semua nilai (210 values) untuk batas kelas konsisten
pooled_padi = df_panel['padi_per_kapita'].dropna().values

quantiles_breaks = mc.Quantiles(pooled_padi, k=5).bins.tolist()
fisher_breaks = mc.FisherJenks(pooled_padi, k=5).bins.tolist()

# Evaluasi Singkat
skewness = pd.Series(pooled_padi).skew()
print("\n--- Evaluasi Klasifikasi ---")
print(f"Skewness Padi per Kapita: {skewness:.2f}")
if abs(skewness) > 1:
    print("Distribusi menceng (skewed). Fisher-Jenks umumnya lebih cocok karena mengelompokkan natural breaks dan mengakomodasi pencilan.")
else:
    print("Distribusi relatif simetris. Quantiles dapat digunakan untuk penyebaran jumlah wilayah yang sama per warna.")

print("Batas Quantiles :", [round(b, 2) for b in quantiles_breaks])
print("Batas Fisher-Jenks:", [round(b, 2) for b in fisher_breaks])

# Hitung jumlah wilayah per kelas per tahun dengan Fisher-Jenks (sebagai contoh)
df_panel['kelas_fisher'] = pd.cut(df_panel['padi_per_kapita'],
                                  bins=[-np.inf] + fisher_breaks,
                                  labels=['K1', 'K2', 'K3', 'K4', 'K5'])
print("\nDistribusi Wilayah di Kelas Fisher-Jenks per Tahun:")
print(pd.crosstab(df_panel['tahun'], df_panel['kelas_fisher']))

# 8. Skala Proportional Symbol
min_pekerja = float(df_panel['penduduk_kerja_pertanian'].min())
max_pekerja = float(df_panel['penduduk_kerja_pertanian'].max())

## Sel 7: Ekspor Data ke JSON/GeoJSON

In [ ]:
# Centroid & Simplify Geometri
gdf_utm = gdf_jateng.to_crs(epsg=32749)
centroids_4326 = gdf_utm.geometry.centroid.to_crs(epsg=4326)
gdf_jateng['centroid_lon'] = centroids_4326.x
gdf_jateng['centroid_lat'] = centroids_4326.y
gdf_jateng['geometry'] = gdf_jateng['geometry'].simplify(tolerance=0.005, preserve_topology=True)

# Ekspor GeoJSON
gdf_jateng[['kode_kabkota', 'nama_kabkota', 'centroid_lon', 'centroid_lat', 'geometry']].to_file('output/jateng_kabkota.geojson', driver='GeoJSON')

# --- PENANGANAN NaN KE NULL UNTUK JSON ---
# Pandas replace np.nan dengan None (python type yang menjadi null di JSON)
df_output_clean = df_output.replace({np.nan: None})
df_moran_clean = pd.DataFrame(hasil_moran_global).replace({np.nan: None})

# Ekspor Tabel Panel JSON
with open('output/data_wilayah_tahun.json', 'w') as f:
    json.dump(df_output_clean.to_dict(orient='records'), f)

# Ekspor Moran Global JSON
with open('output/moran_global.json', 'w') as f:
    json.dump(df_moran_clean.to_dict(orient='records'), f)

# Metadata Baru
metadata = {
    "variabel": {
        "produksi_padi_kg": {"nama": "Produksi Padi (GKG)", "satuan": "kg", "sumber": "BPS"},
        "padi_per_kapita": {"nama": "Produksi padi per kapita", "satuan": "kg/kapita", "sumber": "BPS (Turunan)"},
        "penduduk_kerja_pertanian": {"nama": "Pekerja Pertanian", "satuan": "Jiwa", "sumber": "BPS"}
    },
    "klasifikasi_choropleth": {
        "metode_disarankan": "Fisher-Jenks" if abs(skewness) > 1 else "Quantiles",
        "quantiles_breaks": quantiles_breaks,
        "fisher_jenks_breaks": fisher_breaks
    },
    "skala_simbol": {
        "domain": [min_pekerja, max_pekerja]
    },
    "analisis": {
        "bobot_spasial": "Queen Contiguity (Row-standardized)",
        "permutasi": 999,
        "seed": 42,
        "alpha": 0.05
    }
}
with open('output/metadata.json', 'w') as f:
    json.dump(metadata, f)

print("Berhasil! Semua file telah diekspor (bebas NaN).")

In [ ]:
import matplotlib.pyplot as plt

# 1. Ambil data satu tahun sebagai contoh (misal 2025)
df_2025 = df_output[df_output['tahun'] == 2025]

# 2. Gabungkan geometri ekspor dengan data panel 2025
gdf_plot = gdf_export.merge(df_2025, on='kode_kabkota')

# 3. Siapkan canvas plot
fig, ax = plt.subplots(1, 1, figsize=(12, 8))

# 4. Layer Bawah: Choropleth Padi per Kapita (Fisher-Jenks)
gdf_plot.plot(column='padi_per_kapita',
              scheme='FisherJenks', k=5,
              cmap='YlGn', # Skema warna kuning ke hijau
              linewidth=0.5, edgecolor='gray',
              legend=True,
              legend_kwds={'loc': 'lower left', 'title': 'Padi/Kapita (Fisher-Jenks)'},
              ax=ax)

# 5. Layer Atas: Proportional Symbol (Penduduk Kerja Pertanian)
# Nilai dibagi konstanta agar ukuran lingkarannya muat di peta
sizes = gdf_plot['penduduk_kerja_pertanian'] / 2000
ax.scatter(gdf_plot['centroid_lon'], gdf_plot['centroid_lat'],
           s=sizes, color='orange', alpha=0.6, edgecolor='black',
           label='Pekerja Pertanian')

plt.title('Preview Peta 2025 (Choropleth + Proportional Symbol)')
plt.axis('off') # Hilangkan sumbu koordinat
plt.show()

In [ ]:
from shapely.geometry import Point
import random

print("Mulai membuat titik Dot Density...")

# Kita buat untuk tahun 2025 saja sebagai peta statis penutup narasi
# atau sebagai perbandingan klimaks di Webstory
df_2025 = df_panel[df_panel['tahun'] == 2025]
gdf_2025 = gdf_jateng.merge(df_2025, on='kode_kabkota')

titik_dot_density = []

for idx, row in gdf_2025.iterrows():
    # Hitung berapa titik yang harus dibuat untuk wilayah ini
    n_dots = int(row['produksi_padi_kg'] / 10000000)
    poly = row['geometry']

    # Ambil batas kotak (bounding box) dari poligon wilayah
    minx, miny, maxx, maxy = poly.bounds

    count = 0
    # Looping sebar titik acak sampai jumlahnya pas
    while count < n_dots:
        # Taruh titik secara acak di dalam kotak
        pnt = Point(random.uniform(minx, maxx), random.uniform(miny, maxy))

        # Validasi: Pastikan titik tersebut benar-benar jatuh DI DALAM daratan kabupaten
        # (Bukan di laut atau di kabupaten tetangga yang masuk bounding box)
        if poly.contains(pnt):
            titik_dot_density.append({
                'kode_kabkota': row['kode_kabkota'],
                'nama_kabkota': row['nama_kabkota'],
                'geometry': pnt
            })
            count += 1

# Ubah kumpulan titik menjadi GeoDataFrame
gdf_dots = gpd.GeoDataFrame(titik_dot_density, crs=gdf_jateng.crs)

# Ekspor
output_dots = 'output/dot_density_padi_2025.geojson'
gdf_dots.to_file(output_dots, driver='GeoJSON')

print(f"Selesai! Berhasil membuat {len(gdf_dots)} titik dot density.")
print(f"File diekspor ke: {output_dots}")